# Las dos listas de vulnerabilidades explotadas: el KEV de CISA y la EUVD de ENISA

Cuaderno de lectura de la medición `euvd-kev/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda un artículo en preparación. Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/euvd-kev/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
res = json.loads(texto("resumen.json"))
print("KEV", res["kev_version"], res["kev_entradas"], "entradas · EUVD explotadas", res["euvd_explotadas"], "· fuentes:", res["euvd_fuentes"])
eu = leer("eu-kev.csv")
eu.tail(15)

## Una figura

In [ ]:
v = leer("ventanas-kev.csv").set_index("anio")
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.2))
v.plot.bar(stacked=True, ax=a, rot=0); a.set_title("altas del KEV por año y ventana de plazo"); a.set_ylabel("entradas")
d = eu.dropna(subset=["eu_menos_cisa_dias"]).eu_menos_cisa_dias.astype(int).sort_values().reset_index(drop=True)
b.barh(d.index, d.values, color=["C3" if x < 0 else "C0" for x in d.values]); b.set_xscale("symlog", linthresh=10)
b.set_title("las 44 compartidas: alta EU KEV menos alta CISA KEV (días)"); b.set_yticks([]); plt.tight_layout()